In [1]:
import pandas as pd
import yfinance as yf
from concurrent.futures import ThreadPoolExecutor

# -----------------------------
# Config
# -----------------------------

SP500_SOURCE = "../src/data-pipline/sp500_tickers.csv"# Input CSV with column "Symbol"
SAVE_PATH =    "../src/data-pipline/large_cap_financials.csv"
MAX_WORKERS = 10                          # Number of threads for speed


In [2]:
def preprocess_ticker(ticker):
    """Fix tickers with dots (BRK.B → BRK-B)."""
    return ticker.replace(".", "-").strip()

def is_valid_ticker(ticker):
    """Check if ticker exists in Yahoo Finance."""
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(period="1d")
        return not data.empty
    except:
        return False

def load_sp500_tickers(source=SP500_SOURCE):
    """Load tickers from CSV and preprocess them."""
    df = pd.read_csv(source)
    tickers = df["Ticker"].apply(preprocess_ticker).tolist()
    print(f"[INFO] Loaded {len(tickers)} S&P 500 tickers")
    return tickers

def get_interest_expense_yahoo(ticker):
    try:
        stock = yf.Ticker(ticker)
        # Fetch full income statement (pandas DataFrame)
        income_stmt = stock.financials  # or stock.income_stmt

        # Try multiple possible row names
        for row_name in ["Interest Expense", "InterestExpense", "InterestExpenseNonOperating"]:
            if row_name in income_stmt.index:
                print(income_stmt.loc[row_name].iloc[0])
                return income_stmt.loc[row_name].iloc[0]

        return None
    except Exception as e:
        print(f"[ERROR] {ticker}: {e}")
        return None

def fetch_metrics(ticker):
    """Fetch financial metrics for a ticker, using Yahoo Finance for Interest Expense."""
    try:
        stock = yf.Ticker(ticker)
        info = stock.get_info()
        if not info:
            return None

        # Replace interestExpense with your custom function
        interest_expense = get_interest_expense_yahoo(ticker)

        return {
            "Ticker": ticker,
            "Revenue": info.get("totalRevenue"),
            "Operating_Margin": info.get("operatingMargins"),
            "Net_Margin": info.get("profitMargins"),
            "ROE": info.get("returnOnEquity"),
            "Total_Debt": info.get("totalDebt"),
            "Total_Cash": info.get("totalCash"),
            "EBITDA": info.get("ebitda"),
            "Interest_Expense": interest_expense,  # <-- fully replaced
        }

    except Exception as e:
        print(f"[ERROR] Failed to fetch {ticker}: {e}")
        return None


# -----------------------------
# Main Pipeline
# -----------------------------

def pull_large_cap_dataset(save_path=SAVE_PATH, max_workers=MAX_WORKERS):
    # Step 1 — Load tickers
    tickers = load_sp500_tickers()

    # Step 2 — Filter valid tickers using multithreading
    print("[INFO] Checking valid tickers...")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(lambda t: t if is_valid_ticker(t) else None, tickers))
    valid_tickers = [t for t in results if t]
    print(f"[INFO] {len(valid_tickers)} valid tickers found")

    # Step 3 — Fetch financial metrics for valid tickers (sequentially or threaded if needed)
    print("[INFO] Fetching financial metrics...")
    records = []
    for ticker in valid_tickers:
        record = fetch_metrics(ticker)
        if record:
            records.append(record)

    # Step 4 — Save to CSV
    df = pd.DataFrame(records)
    df.to_csv(save_path, index=False)
    print(f"[SUCCESS] Dataset saved to: {save_path}")
    return df



In [3]:

pull_large_cap_dataset()

[INFO] Loaded 503 S&P 500 tickers
[INFO] Checking valid tickers...


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WBA"}}}
$WBA: possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")


[INFO] 502 valid tickers found
[INFO] Fetching financial metrics...
1191000000.0
6700000.0
559000000.0
2808000000.0
228555000.0
169000000.0
92000000.0
1485000000.0
197000000.0
96000000.0
nan
nan
27117000.0
165619000.0
185838000.0
102000000.0
449000000.0
400000000.0
268000000.0
268000000.0
1124000000.0
2406000000.0
396000000.0
663000000.0
1862800000.0
8252000000.0
462000000.0
1404500000.0
523000000.0
329000000.0
112962000.0
3155000000.0
217000000.0
322227000.0
788000000.0
373000000.0
339000000.0
nan
nan
337000000.0
141000000.0
706000000.0
nan
381300000.0
107000000.0
6759000000.0
171678000.0
nan
455900000.0
486796000.0
226589000.0
117000000.0
7098000.0
198000000.0
293000000.0
90547000000.0
408000000.0
nan
5200000000.0
51000000.0
8509000.0
250300000.0
538000000.0
443688000.0
nan
21295000000.0
2725000000.0
1295000000.0
305000000.0
1947000000.0
3953000000.0
135800000.0
201000000.0
122000000.0
207724000.0
471000000.0
645117000.0
89937000.0
75999000.0
2377000000.0
129815000.0
345000000.0
1482

,Ticker,Revenue,Operating_Margin,Net_Margin,ROE,Total_Debt,Total_Cash,EBITDA,Interest_Expense
0,MMM,24824999936,0.24367,0.13700,0.72921,1.317900e+10,5.188000e+09,6.161000e+09,1.191000e+09
1,AOS,3830099968,0.18631,0.13851,0.28209,2.225000e+08,1.728000e+08,7.843000e+08,6.700000e+06
2,ABT,43842998272,0.19395,0.31880,0.30620,1.297500e+10,7.733000e+09,1.174700e+10,5.590000e+08
3,ABBV,59643998208,0.35497,0.04004,1.37961,6.884900e+10,5.671000e+09,2.951900e+10,2.808000e+09
4,ACN,69672976384,0.15220,0.11021,0.25509,8.182866e+09,1.148467e+10,1.222253e+10,2.285550e+08
...,...,...,...,...,...,...,...,...,...
497,XYL,8894000128,0.15653,0.10659,0.08555,2.089000e+09,1.191000e+09,1.854000e+09,4.400000e+07
498,YUM,8061000192,0.34411,0.17951,NaN,1.248400e+10,1.045000e+09,2.819000e+09,4.890000e+08
499,ZBRA,5255000064,0.14621,0.09743,0.14286,2.360000e+09,1.053000e+09,9.740000e+08,NaN
500,ZBH,8010899968,0.17003,0.10053,0.06421,8.237000e+09,1.306100e+09,2.640500e+09,2.180000e+08


In [41]:
# Step 2 — Filter valid tickers
valid_tickers = [t for t in df["Ticker"] if is_valid_ticker(t)]
print(f"[INFO] {len(valid_tickers)} valid tickers found")

$BRK.B: possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")
$BF.B: possibly delisted; no price data found  (period=1d)
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WBA"}}}
$WBA: possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")


[INFO] 500 valid tickers found


In [ ]:
# Save to CSV
df.to_csv("../data/sp500_tickers.csv", index=False)
print(f"Loaded {len(tickers)} tickers")

### load tickers from saved CSV files


In [39]:
def load_sp500_tickers():
    df=pd.read_csv("../data/sp500_tickers.csv")
    tickers = df["Ticker"].tolist()
    print(f"[INFO] Loaded {len(tickers)} S&P 500 tickers.")
    return tickers
tickers=load_sp500_tickers()


[INFO] Loaded 503 S&P 500 tickers.


In [ ]:
def is_valid_ticker(ticker):
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(period="1d")
        return not data.empty
    except:
        return False

def fetch_metrics(ticker):
    """Extract required financial metrics from Yahoo Finance."""
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        return {
            "Ticker": ticker,
            "Revenue": info.get("totalRevenue"),
            "Operating_Margin": info.get("operatingMargins"),
            "Net_Margin": info.get("profitMargins"),
            "ROE": info.get("returnOnEquity"),
            "Total_Debt": info.get("totalDebt"),
            "Total_Cash": info.get("totalCash"),
            "EBITDA": info.get("ebitda"),
            "Interest_Expense": info.get("interestExpense"),
        }

    except Exception as e:
        print(f"[ERROR] Failed to fetch {ticker}: {e}")
        return None


def pull_large_cap_dataset(save_path="data/large_cap_financials.csv"):
    tickers = load_sp500_tickers()
    records = []

    for ticker in tickers:
        record = fetch_metrics(ticker)
        if record:
            records.append(record)

    df = pd.DataFrame(records)
    df.to_csv(save_path, index=False)
    print(f"[SUCCESS] Saved dataset to: {save_path}")
    return df

In [3]:
pull_large_cap_dataset()

[INFO] Loaded 503 S&P 500 tickers.


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WBA"}}}


OSError: Cannot save file into a non-existent directory: 'data'